# Lab 1: Goal-based agent
Source: [agents_lab.pdf](agents_lab.pdf). Tasks follow the sheet's order.

LLM assistance: Codex prepared the code and concise written answers; saved outputs record executed checks.


## Task 1: Understand the problem
1. **Environment:** a static warehouse grid containing shelves and free cells.
2. **Goal:** reach `G` from `S` without crossing a shelf.
3. **Actions:** move up, down, left, or right by one cell.
4. **Information:** map, current position, goal, frontier, visited positions, and parent links.
5. **Goal-based:** actions come from a plan to reach an explicit destination.

**Think about it:** BFS still finds a shortest path in a larger unit-cost grid, but its time and memory grow with reachable cells. A* can reduce exploration using a suitable heuristic; changing shelves would require replanning.


## Task 2: Design the agent
State: `(row, column)`. Goal test: current position equals `G`.
BFS explores legal moves in increasing path length; parent links reconstruct the plan.

```text
Warehouse map + current state + goal
                 |
                 v
       BFS decision component
                 |
                 v
         Planned next move
                 |
                 v
         Updated position ----> goal test
```


## Task 3: Prompt, implementation, and testing
**Implementation prompt used for this LLM-assisted submission:**

> Implement a readable Python goal-based agent for the exact warehouse map below. Use a two-dimensional grid, four unit-cost moves, and BFS to find a shortest collision-free path from S to G. Preserve the printed map; pad missing cells in shorter rows with obstacles. Print the path, move count, and map overlay, or No path found. Explain the choice of BFS briefly. Verify endpoints, every move, obstacle avoidance, and an unreachable goal with executable assertions.

BFS is appropriate because all moves cost one. Coordinates below are zero-based.
The sheet's border rows have 21 characters and internal rows have 20; right-padding with `#` preserves all supplied cells.


In [1]:
from collections import deque

warehouse = [
    "#####################",
    "#S....#............G#",
    "#.##....##########..#",
    "#....##.............#",
    "#.######.###.#.###..#",
    "#........#..........#",
    "#####################",
]
width = max(map(len, warehouse))
grid = [row.ljust(width, "#") for row in warehouse]


def locate(grid, symbol):
    positions = [(r, c) for r, row in enumerate(grid)
                 for c, value in enumerate(row) if value == symbol]
    if len(positions) != 1:
        raise ValueError(f"Expected exactly one {symbol}")
    return positions[0]


def neighbors(grid, position):
    row, column = position
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        r, c = row + dr, column + dc
        if 0 <= r < len(grid) and 0 <= c < len(grid[r]) and grid[r][c] != "#":
            yield r, c


def breadth_first_path(grid, start, goal):
    frontier = deque([start])
    parent = {start: None}
    while frontier:
        position = frontier.popleft()
        if position == goal:
            path = []
            while position is not None:
                path.append(position)
                position = parent[position]
            return path[::-1]
        for successor in neighbors(grid, position):
            if successor not in parent:
                parent[successor] = position
                frontier.append(successor)
    return None


def show_path(grid, path):
    drawing = [list(row) for row in grid]
    if path:
        for row, column in path[1:-1]:
            drawing[row][column] = "*"
    print("\n".join("".join(row) for row in drawing))


start, goal = locate(grid, "S"), locate(grid, "G")
path = breadth_first_path(grid, start, goal)
if path is None:
    print("No path found")
else:
    print("Path:", path)
    print("Moves:", len(path) - 1)
show_path(grid, path)


Path: [(1, 1), (1, 2), (1, 3), (1, 4), (2, 4), (2, 5), (2, 6), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (1, 16), (1, 17), (1, 18), (1, 19)]
Moves: 20
#####################
#S***.#************G#
#.##****##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################


In [2]:
assert path is not None and path[0] == start and path[-1] == goal
assert all(grid[r][c] != "#" for r, c in path)
assert all(b in list(neighbors(grid, a)) for a, b in zip(path, path[1:]))
blocked = ["#####", "#S#G#", "#####"]
impossible = breadth_first_path(blocked, locate(blocked, "S"), locate(blocked, "G"))
assert impossible is None
print("Original path: valid")
print("Blocked goal: No path found")
print("All checks passed")


Original path: valid
Blocked goal: No path found
All checks passed


**Required answers:**
1. The initial submitted implementation executed successfully and passed these checks.
2. A precise prompt should specify bounds, malformed map handling, failure behavior, and tests.
3. The chosen algorithm is BFS.
4. Equal move costs make BFS a simple complete shortest-path method for this finite grid.
